In [1]:
# Install
!pip install torch -q

In [2]:
# Import
import torch
import torch.nn as nn
import math

In [3]:
# Group Query Attention
class SimpleGQA(nn.Module):
    def __init__(self, d_model=256, n_heads=8, n_kv_heads=2):
        super().__init__()

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.group = n_heads // n_kv_heads
        self.head_dim = d_model // n_heads

        # Linear layers
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        # Queries (full heads)
        q = self.q(x).view(B, T, self.n_heads, self.head_dim)

        # Keys & Values (fewer heads)
        k = self.k(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.v(x).view(B, T, self.n_kv_heads, self.head_dim)

        # Repeat K,V to match Q heads
        k = k.repeat_interleave(self.group, dim=2)
        v = v.repeat_interleave(self.group, dim=2)

        # Change shape for attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Attention
        score = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        weight = torch.softmax(score, dim=-1)
        out = weight @ v

        # Merge heads
        out = out.transpose(1, 2).reshape(B, T, C)

        return self.out(out)

In [4]:
# Test
x = torch.randn(2, 100, 256)

model = SimpleGQA()
y = model(x)

print("Output shape:", y.shape)

Output shape: torch.Size([2, 100, 256])


In [5]:
# Memory comparisson
n_heads = 8
n_kv_heads = 2
seq_len = 100
head_dim = 32

# Without GQA
normal = n_heads * seq_len * head_dim

# With GQA
gqa = n_kv_heads * seq_len * head_dim

print("Normal KV memory:", normal)
print("GQA KV memory:", gqa)
print("Saved:", normal // gqa, "times less memory")

Normal KV memory: 25600
GQA KV memory: 6400
Saved: 4 times less memory
